# 3. Statistical inference
Read reports/analysis_plan.md first. Four alpha=.05 tests; thresholds were fixed before comparing groups. Computations below independently verify statistics from actual saved sufficient summaries.

In [1]:
from pathlib import Path
import json
import pandas as pd
root = Path.cwd()
if not (root / 'outputs').exists(): root = root.parent
assert (root / 'outputs/results.json').exists(), 'Run scripts/run_all.py first'
def read(name): return json.loads((root / 'outputs' / name).read_text(encoding='utf-8'))


In [2]:
from scipy import stats
import numpy as np
r=read('results.json')
for v in r[:2]:
    ref=stats.ttest_ind_from_stats(v['mean_a'],v['sd_a'],v['n_a'],v['mean_b'],v['sd_b'],v['n_b'],equal_var=False)
    assert np.isclose(ref.statistic,v['statistic'])
    assert v['ci_low'] < v['estimate'] < v['ci_high']
    print(v['test'], 'difference:',v['estimate'], '95% CI:',[v['ci_low'],v['ci_high']], 'p:',v['p_display'])

int_rate difference: 3.087425342314962 95% CI: [3.0669798058106066, 3.107870878819317] p: < 2.23e-308 (floating-point underflow)
dti difference: 2.3595669702060853 95% CI: [2.3102744733704106, 2.40885946704176] p: < 2.23e-308 (floating-point underflow)


In [3]:
table=pd.read_csv(root/'outputs/tables/grade_outcome.csv',index_col=0)
chi,p,df,expected=stats.chi2_contingency(table.values,correction=False)
assert np.isclose(chi,r[2]['statistic']) and df==r[2]['df']
assert expected.min()>=5
print('Grade chi-square:',chi,'df:',df,'Cramers V:',r[2]['estimate'],'CI:',[r[2]['ci_low'],r[2]['ci_high']])

Grade chi-square: 92284.25105933327 df: 6 Cramers V: 0.2619103197696805 CI: [0.26028717063124773, 0.2634500553863387]


In [4]:
term=r[3]
assert np.isclose(term['statistic']**2,term['chi2_crosscheck'])
print('60 months:',term['events_a'],'/',term['n_a'],'=',term['proportion_a'])
print('36 months:',term['events_b'],'/',term['n_b'],'=',term['proportion_b'])
print('Difference in percentage points:',100*term['estimate'])
print('95% CI in percentage points:',[100*term['ci_low'],100*term['ci_high']])

60 months: 105307 / 324567 = 0.3244538107694251
36 months: 163252 / 1020743 = 0.15993447910002812
Difference in percentage points: 16.451933166939696
95% CI in percentage points: [16.27587038211621, 16.627995951763186]


In [5]:
for v in r:
    print(v['test'],v['decision'],'Holm p:',v['holm_p_display'],'Threshold:',v['threshold'],v['units'],'Practical:',v['practical'])
s=read('sensitivity.json')
for key,v in s.items():
    print(key,'capped sensitivity difference:',v['result']['estimate'],'CI:',[v['result']['ci_low'],v['result']['ci_high']])

int_rate Reject H0 Holm p: < 2.23e-308 (floating-point underflow) Threshold: 1.0 percentage points Practical: CI beyond positive threshold
dti Reject H0 Holm p: < 2.23e-308 (floating-point underflow) Threshold: 2.0 percentage points Practical: CI beyond positive threshold
grade Reject H0 Holm p: < 2.23e-308 (floating-point underflow) Threshold: 0.1 Cramers V Practical: CI beyond positive threshold
term Reject H0 Holm p: < 2.23e-308 (floating-point underflow) Threshold: 0.02 proportion (60 minus 36 months) Practical: CI beyond positive threshold
int_rate capped sensitivity difference: 3.037007036409765 CI: [3.017201329938703, 3.056812742880827]
dti capped sensitivity difference: 2.2907514383703784 CI: [2.254456213171556, 2.3270466635692006]


## Defensible conclusion
Compare intervals with practical thresholds, not just zero. Underflowed p-values are reported as bounds, never as zero. Holm-adjusted decisions do not remove selection, censoring, dependence or confounding. Capped sensitivity changes the estimand. The chi-square cross-check for term is not a fifth test. Bootstrap uncertainty for Cramer's V assumes IID observations. See the technical report for formulas and alternative tests.